In [1]:
import json
import re
import subprocess
from pathlib import Path

import polars as pl
from linkml_runtime import SchemaView

DATA = Path("../../benchmark_data")

/home/piotr/neverblink/research/linkml-benchmark-schemas/analysis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_imports(path: Path) -> list[str]:
    """Top-level `imports:` list of a schema file (lightweight, no full parse)."""
    out, in_block = [], False
    for line in path.read_text().splitlines():
        if re.match(r"^imports:\s*$", line):
            in_block = True
            continue
        if in_block:
            m = re.match(r"^\s*-\s*(.+?)\s*$", line)
            if m:
                val = m.group(1).split("#")[0].strip()  # drop inline comments
                if val:
                    out.append(val)
            elif line.strip() and not line[0].isspace():
                break  # dedented to a new top-level key
    return out


def resolve(imp: str, base: Path) -> Path | None:
    if ":" in imp and not imp.startswith((".", "/")):
        return None  # built-in, e.g. linkml:types
    p = base / imp
    for cand in (p.with_suffix(".yaml"), p.with_suffix(".yml"), p):
        if cand.is_file():
            return cand.resolve()
    return None


def import_closure(main: Path) -> dict[Path, int]:
    """{file: size_bytes} for main.yaml and everything it (transitively) imports."""
    seen: dict[Path, int] = {}
    stack = [main.resolve()]
    while stack:
        f = stack.pop()
        if f in seen:
            continue
        seen[f] = f.stat().st_size
        for imp in read_imports(f):
            r = resolve(imp, f.parent)
            if r and r not in seen:
                stack.append(r)
    return seen

In [3]:
rows = []
for d in sorted(p for p in DATA.iterdir() if p.is_dir()):
    files = import_closure(d / "main.yaml")
    sv = SchemaView(str(d / "main.yaml"))
    classes = sv.all_classes(imports=True)
    attrs = sum(len(sv.class_induced_slots(cn, imports=True)) for cn in classes)
    rows.append({
        "dataset": d.name,
        "files": len(files),
        "size_kb": sum(files.values()) / 1024,
        "classes": len(classes),
        "attributes": attrs,
    })

df = pl.DataFrame(rows).sort("size_kb")
df

dataset,files,size_kb,classes,attributes
str,i64,f64,i64,i64
"""brigde2ai_model_card""",1,39.518555,38,160
"""include""",1,56.25293,10,149
"""sssom""",1,59.420898,9,110
"""ai-atlas-nexus""",10,69.510742,99,1781
"""chem-dcat-ap""",5,117.360352,89,889
…,…,…,…,…
"""nmdc_microbiome""",14,558.848633,80,1655
"""crdch""",1,1064.801758,41,335
"""cdm""",37,2237.902344,779,3245


In [4]:
def to_latex(df: pl.DataFrame) -> str:
    esc = lambda s: s.replace("_", r"\_")
    lines = [
        r"\begin{table}[t]",
        r"  \centering",
        r"  \caption{Benchmark dataset statistics. \emph{Files} and \emph{Size} "
        r"cover the import closure of \texttt{main.yaml}; \emph{Classes} and "
        r"\emph{Attributes} are the materialised (induced) totals.}",
        r"  \label{tab:datasets}",
        r"  \begin{tabular}{lrrrr}",
        r"    \toprule",
        r"    Dataset & Files & Size (KiB) & Classes & Attributes \\",
        r"    \midrule",
    ]
    for r in df.iter_rows(named=True):
        lines.append(
            f"    {esc(r['dataset'])} & {r['files']} & {r['size_kb']:.1f} "
            f"& {r['classes']} & {r['attributes']} \\\\"
        )
    tot = df.select(
        pl.col("files").sum(), pl.col("size_kb").sum(),
        pl.col("classes").sum(), pl.col("attributes").sum(),
    ).row(0)
    lines += [
        r"    \midrule",
        f"    Total & {tot[0]} & {tot[1]:.1f} & {tot[2]} & {tot[3]} \\\\",
        r"    \bottomrule",
        r"  \end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)


latex = to_latex(df)
Path("../tables/dataset_stats.tex").write_text(latex + "\n")
print(latex)

\begin{table}[t]
  \centering
  \caption{Benchmark dataset statistics. \emph{Files} and \emph{Size} cover the import closure of \texttt{main.yaml}; \emph{Classes} and \emph{Attributes} are the materialised (induced) totals.}
  \label{tab:datasets}
  \begin{tabular}{lrrrr}
    \toprule
    Dataset & Files & Size (KiB) & Classes & Attributes \\
    \midrule
    brigde2ai\_model\_card & 1 & 39.5 & 38 & 160 \\
    include & 1 & 56.3 & 10 & 149 \\
    sssom & 1 & 59.4 & 9 & 110 \\
    ai-atlas-nexus & 10 & 69.5 & 99 & 1781 \\
    chem-dcat-ap & 5 & 117.4 & 89 & 889 \\
    fluxnova-bpm & 17 & 244.3 & 258 & 2765 \\
    iso27001 & 1 & 251.7 & 35 & 781 \\
    nmdc\_microbiome & 14 & 558.8 & 80 & 1655 \\
    crdch & 1 & 1064.8 & 41 & 335 \\
    cdm & 37 & 2237.9 & 779 & 3245 \\
    d3fend & 1 & 2591.3 & 4366 & 5250 \\
    tc57cim & 1 & 2952.3 & 1528 & 34172 \\
    \midrule
    Total & 90 & 10243.2 & 7332 & 51292 \\
    \bottomrule
  \end{tabular}
\end{table}


## Transposed table (datasets as columns, alphabetical)

In [5]:
def to_latex_wide(df: pl.DataFrame) -> str:
    """Transposed table: metrics as rows, datasets as columns (alphabetical).

    Dataset headers are rotated 90 deg to fit 12 columns; requires the graphicx
    package in the LaTeX preamble (for \rotatebox).
    """
    df = df.sort("dataset")
    esc = lambda s: s.replace("_", r"\_")
    names = df["dataset"].to_list()
    ncol = len(names)

    def row(label, key, fmt):
        vals = " & ".join(fmt(v) for v in df[key].to_list())
        return f"    {label} & {vals} \\\\"

    header = "     & " + " & ".join(
        rf"\rotatebox{{90}}{{{esc(n)}}}" for n in names
    ) + r" \\"

    lines = [
        r"\begin{table}[t]",
        r"  \centering",
        r"  \caption{Benchmark dataset statistics (transposed). Datasets are "
        r"ordered alphabetically; see Table~\ref{tab:datasets} for column "
        r"definitions.}",
        r"  \label{tab:datasets-wide}",
        rf"  \begin{{tabular}}{{l{'r' * ncol}}}",
        r"    \toprule",
        header,
        r"    \midrule",
        row("Files", "files", str),
        row("Size (KiB)", "size_kb", lambda v: f"{v:.1f}"),
        row("Classes", "classes", str),
        row("Attributes", "attributes", str),
        r"    \bottomrule",
        r"  \end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)


latex_wide = to_latex_wide(df)
Path("../tables/dataset_stats_wide.tex").write_text(latex_wide + "\n")
print(latex_wide)


\begin{table}[t]
  \centering
  \caption{Benchmark dataset statistics (transposed). Datasets are ordered alphabetically; see Table~\ref{tab:datasets} for column definitions.}
  \label{tab:datasets-wide}
  \begin{tabular}{lrrrrrrrrrrrr}
    \toprule
     & \rotatebox{90}{ai-atlas-nexus} & \rotatebox{90}{brigde2ai\_model\_card} & \rotatebox{90}{cdm} & \rotatebox{90}{chem-dcat-ap} & \rotatebox{90}{crdch} & \rotatebox{90}{d3fend} & \rotatebox{90}{fluxnova-bpm} & \rotatebox{90}{include} & \rotatebox{90}{iso27001} & \rotatebox{90}{nmdc\_microbiome} & \rotatebox{90}{sssom} & \rotatebox{90}{tc57cim} \\
    \midrule
    Files & 10 & 1 & 37 & 5 & 1 & 1 & 17 & 1 & 1 & 14 & 1 & 1 \\
    Size (KiB) & 69.5 & 39.5 & 2237.9 & 117.4 & 1064.8 & 2591.3 & 244.3 & 56.3 & 251.7 & 558.8 & 59.4 & 2952.3 \\
    Classes & 99 & 38 & 779 & 89 & 41 & 4366 & 258 & 10 & 35 & 80 & 9 & 1528 \\
    Attributes & 1781 & 160 & 3245 & 889 & 335 & 5250 & 2765 & 149 & 781 & 1655 & 110 & 34172 \\
    \bottomrule
  \end{tabula

## Validation: `SchemaView` vs. `linkml-scala generate linkml`

In [6]:
def scala_counts(main: Path):
    r = subprocess.run(
        ["linkml-scala", "generate", "linkml",
         "--pruning-mode", "skip", "--format", "json", str(main)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        return None  # linkml-scala JSON encoder crashes on d3fend / nmdc
    schema = json.loads(r.stdout)
    cls = schema.get("classes") or {}
    return len(cls), sum(len(c.get("attributes") or {}) for c in cls.values())


checked, skipped = 0, []
for r in df.iter_rows(named=True):
    sc = scala_counts(DATA / r["dataset"] / "main.yaml")
    if sc is None:
        skipped.append(r["dataset"])
        continue
    assert sc == (r["classes"], r["attributes"]), (r["dataset"], sc)
    checked += 1

print(f"linkml-scala agrees with SchemaView on {checked} datasets")
print(f"linkml-scala crashed (skipped): {skipped}")

linkml-scala agrees with SchemaView on 10 datasets
linkml-scala crashed (skipped): ['nmdc_microbiome', 'd3fend']
